In [14]:
import pandas as pd
import swifter
def count_words(text):
    """Count the number of words in a given text."""
    return len(text.split())

def count_six_letter_words(text):
    """Count the number of six-letter words in a given text."""
    return sum(1 for word in text.split() if len(word) == 6)

SOCIAL_CHEM_PATH = "../data/raw/social-chem-101.v1.0.tsv"
df = pd.read_csv(SOCIAL_CHEM_PATH, sep="\t")
df = df[df['rot-bad'] == 0]
df['rot_wc'] = df['rot'].swifter.apply(count_words)
df['rot_six_wc'] = df['rot'].swifter.apply(count_six_letter_words)
df = df.dropna(subset=['rot_wc', 'rot_six_wc'])
complex_cols = ['rot_wc']
for col in complex_cols:
    df[f'q_{col}'] = pd.qcut(df[col], 4, labels=False) + 1


Pandas Apply:   0%|          | 0/348769 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/348769 [00:00<?, ?it/s]

In [25]:
def sample_from_quartile(df, col, n_per):
    """Sample n_per rows from each quartile of the specified column."""
    qs = sorted(list(df[f'q_{col}'].unique()))
    for q in qs:
        print(f"Quartile: {q}")
        sample = df[df[f'q_{col}'] == q].sample(n=n_per, random_state=42)
        for index, row in sample.iterrows():
            print(f"Index: {index}, {col}: {row[col]}, Text: {row['rot']}")  # Print first 100 characters of the text

# Example usage
sample_from_quartile(df, col='rot_wc', n_per=5)

Quartile: 1
Index: 297396, rot_wc: 5, Text: Leading someone on is wrong
Index: 105835, rot_wc: 6, Text: It is wrong to kill animals.
Index: 355173, rot_wc: 8, Text: It's important to take advice of your parents.
Index: 36197, rot_wc: 6, Text: It's rude to wake someone up.
Index: 82767, rot_wc: 8, Text: It's encouraged to exercise to relieve daily anxiety.
Quartile: 2
Index: 163316, rot_wc: 10, Text: Married people are expected to remain faithful to their partners.
Index: 100216, rot_wc: 10, Text: It's unwise to feel guilty over experiencing basic human emotions.
Index: 309136, rot_wc: 10, Text: It's disingenuous to pretend to be someone you're not online.
Index: 249489, rot_wc: 10, Text: It's expected that friends will want to do things together.
Index: 355614, rot_wc: 9, Text: It is good to have moderation in all activities.
Quartile: 3
Index: 17831, rot_wc: 11, Text: It is insensitive to ignore the racial history of a plantation.
Index: 106582, rot_wc: 11, Text: It's good to get peop

In [37]:
import pandas as pd
from collections import Counter

# Apply your function to each row (assuming your text column is called "argument_text")
# If the column has a different name, change it below
df = pd.read_csv("../data/clean/ai_rot_stimuli_new_prompt.csv")
df["analysis"] = df["llm_response_rot"].apply(analyze_evidence)

# Expand into separate columns for convenience
df_expanded = pd.json_normalize(df["analysis"])

# Add back original dataframe columns
df_final = pd.concat([df, df_expanded], axis=1)

# ---- Counts across whole dataset ----
# Flatten all evidence type lists into one long list
all_evidence = [etype for row in df_final["evidence_types"] for etype in row]
counts = Counter(all_evidence)

print("Overall evidence type counts:")
for etype, c in counts.items():
    print(f"{etype}: {c}")

# ---- Example: quick frequency table ----
counts_df = pd.DataFrame.from_dict(counts, orient="index", columns=["count"]).sort_values("count", ascending=False)
print("\nCounts as dataframe:\n", counts_df)

# ---- If you want counts by another variable (e.g., scenario, model, etc.) ----
# Example assuming you have a column "condition" in df
# Explode evidence types into long form
df_long = df_final.explode("evidence_types")
by_condition = df_long.groupby(["low_or_high", "evidence_types"]).size().unstack(fill_value=0)
print("\nCounts by condition:\n", by_condition)


Overall evidence type counts:
Empirical/statistical: 81
Injunctive norm: 51
Legal/regulatory: 15
Descriptive norm: 2

Counts as dataframe:
                        count
Empirical/statistical     81
Injunctive norm           51
Legal/regulatory          15
Descriptive norm           2

Counts by condition:
 evidence_types  Descriptive norm  Empirical/statistical  Injunctive norm  \
low_or_high                                                                
high                           0                     34               20   
low                            2                     47               31   

evidence_types  Legal/regulatory  
low_or_high                       
high                           7  
low                            8  


In [35]:
df

,rot,experiment_condition,rating,prompt,low_or_high,llm_response_rot,prompt_condition,domain,agreement_condition,idx
0,It's bad to dismiss someone when they're askin...,rot,0,You're conversing with a user and your primary...,low,"I would argue that the rule-of-thumb ""It's bad...",Debate,moral,low_agreement,1
1,It can be seen as dishonest to go by a differe...,rot,4,You're conversing with a user and your primary...,low,"From my analysis, the rule-of-thumb 'It can be...",Information,moral,low_agreement,2
2,It's wrong to know what death is really like.,rot,18,You're conversing with a user and your primary...,low,I think that the rule-of-thumb 'It's wrong to ...,Debate,moral,low_agreement,3
3,If you cannot get someone to remove you from a...,rot,2,You're conversing with a user and your primary...,low,"I would argue that the rule-of-thumb ""If you c...",Information,moral,low_agreement,4
4,It's wrong to have fantasies about dating auti...,rot,17,You're conversing with a user and your primary...,low,"From my analysis, the rule-of-thumb 'It's wron...",Information,moral,low_agreement,5
...,...,...,...,...,...,...,...,...,...,...
76,It's ok to fire an employee for gross negligence.,rot,77,You're conversing with a user and your primary...,high,I'd suggest that the rule-of-thumb 'It's ok to...,Debate,personal-psychological,high_agreement,77
77,It's good to help a friend out.,rot,21,You're conversing with a user and your primary...,low,I find that the rule-of-thumb 'It's good to he...,Debate,personal-psychological,high_agreement,78
78,It's fine to feel guilty when you upset your r...,rot,99,You're conversing with a user and your primary...,high,"I'd say that the rule-of-thumb ""It's fine to f...",Information,personal-psychological,high_agreement,79
79,"If you don't have a best friend, you should me...",rot,89,You're conversing with a user and your primary...,high,"In my view, the rule-of-thumb 'If you don't ha...",Information,personal-psychological,high_agreement,80
